# Hugging Face Text Classification Workflow

## Notebook Contract
- **Objective:** demonstrate a reproducible end-to-end text-classification workflow (synthetic data -> train -> evaluate -> export artifacts).
- **Inputs:** local repository files and synthetic dataset generated in this notebook.
- **Outputs:** sample dataset under `data/raw/`, plus CLI commands for model/eval artifacts under `artifacts/` and `reports/`.
- **Expected runtime:** under 5 minutes for the cells in this notebook (training is shown as a command, not executed by default).
- **Scope:** this notebook is a development/demo orchestrator; production logic must stay in `src/hf_finetuning_lab`.


## 1) Setup and Reproducibility


In [ ]:
from __future__ import annotations

import os
import platform
import random
import sys
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'Seed: {SEED}')


## 2) Parameters


In [ ]:
ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'raw' / 'support_tickets.csv'
MODEL_DIR = ROOT / 'artifacts' / 'models' / 'support-triage-notebook'
EVAL_PATH = ROOT / 'reports' / 'sample_run' / 'evaluation.json'
PRED_PATH = ROOT / 'reports' / 'sample_run' / 'predictions.csv'

MODEL_NAME = 'distilbert-base-uncased'
TEXT_COL = 'text'
LABEL_COL = 'label'
ROWS = 500
EPOCHS = 1
BATCH_SIZE = 8
ALLOW_HF_DOWNLOADS = False

print(f'Root: {ROOT}')
print(f'Data path: {DATA_PATH}')
print(f'Model dir: {MODEL_DIR}')


## 3) Data Loading and Validation


In [ ]:
import pandas as pd

from hf_finetuning_lab.sample_data import write_sample_data

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_sample_data(output=DATA_PATH, rows=ROWS, seed=SEED)
df = pd.read_csv(DATA_PATH)
df.head()


## 4) EDA


In [ ]:
df[LABEL_COL].value_counts().sort_values(ascending=False)


## 5) Baseline and Fine-tuning (CLI)

Training is intentionally run via CLI for stability and parity with CI/docs.


In [ ]:
if not ALLOW_HF_DOWNLOADS:
    print('Skipping training execution because ALLOW_HF_DOWNLOADS=False')

train_cmd = (
    f"poetry run hf-lab train --input {DATA_PATH} --output-dir {MODEL_DIR} "
    f"--model-name {MODEL_NAME} --text-col {TEXT_COL} --label-col {LABEL_COL} "
    f"--epochs {EPOCHS} --batch-size {BATCH_SIZE}"
)
print(train_cmd)


## 6) Evaluation and Error Analysis (CLI)


In [ ]:
eval_cmd = (
    f"poetry run hf-lab evaluate --model-dir {MODEL_DIR} --input {DATA_PATH} "
    f"--output {EVAL_PATH} --text-col {TEXT_COL} --label-col {LABEL_COL}"
)
pred_cmd = (
    f"poetry run hf-lab predict --model-dir {MODEL_DIR} --input {DATA_PATH} "
    f"--output {PRED_PATH} --text-col {TEXT_COL}"
)
print(eval_cmd)
print(pred_cmd)


## 7) Export Artifacts Checklist

- Model directory exists after training (`artifacts/models/...`).
- Evaluation JSON exists and includes metrics + confusion matrix.
- Predictions CSV exists.
- Model card includes synthetic-data limitation statements.
